In [ ]:
import os
import torch
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image
import torch.nn as nn
import torch.nn.functional as F

# Allow MPS memory override
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

# Device
DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Paths
BASE_PATH = "/Users/imamahasan/MyData/Code/LoDoPaB-CT"
TRAIN_OBS_DIR = os.path.join(BASE_PATH, "observation_train")
TRAIN_GT_DIR = os.path.join(BASE_PATH, "ground_truth_train")
CHECKPOINT_DIR = './checkpoints'
OUTPUT_DIR = './outputs'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
BATCH_SIZE = 1
LR = 1e-4
EPOCHS = 50
TARGET_SIZE = (362, 362)  # Fixed target size

# Dataset with resizing
class LoDoPaBDataset(Dataset):
    def __init__(self, obs_dir, gt_dir, target_size=TARGET_SIZE):
        self.obs_files = sorted(os.listdir(obs_dir))
        self.obs_dir = obs_dir
        self.gt_dir = gt_dir
        self.target_size = target_size

    def __len__(self):
        return len(self.obs_files)

    def __getitem__(self, idx):
        obs_path = os.path.join(self.obs_dir, self.obs_files[idx])
        gt_path = os.path.join(self.gt_dir, self.obs_files[idx].replace("observation", "ground_truth"))

        with h5py.File(obs_path, 'r') as f_obs, h5py.File(gt_path, 'r') as f_gt:
            obs = f_obs['data'][:].astype(np.float32)
            gt = f_gt['data'][:].astype(np.float32)

        # Normalize to [-1, 1]
        obs = (obs - obs.min()) / (obs.max() - obs.min()) * 2 - 1
        gt = (gt - gt.min()) / (gt.max() - gt.min()) * 2 - 1

        # Convert to tensor and resize
        obs_tensor = torch.from_numpy(obs[0]).unsqueeze(0).unsqueeze(0)
        gt_tensor = torch.from_numpy(gt[0]).unsqueeze(0).unsqueeze(0)
        
        # Resize to target dimensions
        obs_tensor = F.interpolate(obs_tensor, size=self.target_size, mode='bilinear', align_corners=False).squeeze(0)
        gt_tensor = F.interpolate(gt_tensor, size=self.target_size, mode='bilinear', align_corners=False).squeeze(0)

        return obs_tensor, gt_tensor

# Generator with exact 362×362 output
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        base = 48
        
        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, base, 3, stride=1, padding=1),
            nn.BatchNorm2d(base),
            nn.ReLU()
        )
        self.enc2 = nn.Sequential(
            nn.Conv2d(base, base*2, 3, stride=2, padding=1),
            nn.BatchNorm2d(base*2),
            nn.ReLU()
        )
        self.enc3 = nn.Sequential(
            nn.Conv2d(base*2, base*4, 3, stride=2, padding=1),
            nn.BatchNorm2d(base*4),
            nn.ReLU()
        )
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(base*4, base*4, 3, stride=1, padding=1),
            nn.BatchNorm2d(base*4),
            nn.ReLU()
        )
        
        # Decoder with precise output dimensions
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(base*4, base*2, 3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(base*2),
            nn.ReLU()
        )
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(base*2, base, 3, stride=2, padding=1, output_padding=0),  # Adjusted for 362 output
            nn.BatchNorm2d(base),
            nn.ReLU()
        )
        self.dec1 = nn.Sequential(
            nn.Conv2d(base, base, 3, stride=1, padding=1),
            nn.BatchNorm2d(base),
            nn.ReLU()
        )
        
        # Final layer with crop if needed
        self.final = nn.Sequential(
            nn.Conv2d(base, 1, kernel_size=1),
            nn.Tanh()
        )

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)        # [1, 48, 362, 362]
        e2 = self.enc2(e1)       # [1, 96, 181, 181]
        e3 = self.enc3(e2)       # [1, 192, 91, 91]
        
        # Bottleneck
        b = self.bottleneck(e3)  # [1, 192, 91, 91]
        
        # Decoder
        d3 = self.dec3(b)        # [1, 96, 182, 182]
        d2 = self.dec2(d3)       # [1, 48, 362, 362]
        d1 = self.dec1(d2)       # [1, 48, 362, 362]
        
        # Final output
        output = self.final(d1)  # [1, 1, 362, 362]
        
        # Ensure exact dimensions (crop if necessary)
        if output.size(2) > 362 or output.size(3) > 362:
            output = output[:, :, :362, :362]
        
        return output

# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        base = 48
        self.model = nn.Sequential(
            nn.Conv2d(1, base, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(base, base*2, 4, stride=2, padding=1),
            nn.BatchNorm2d(base*2),
            nn.LeakyReLU(0.2),
            nn.Conv2d(base*2, base*4, 4, stride=2, padding=1),
            nn.BatchNorm2d(base*4),
            nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(base*4, 1)
        )

    def forward(self, x):
        return self.model(x)

# Training
def train():
    # Initialize models
    G = Generator().to(DEVICE)
    D = Discriminator().to(DEVICE)
    
    # Optimizers
    opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
    
    # Loss functions
    criterion_mse = nn.MSELoss()
    criterion_bce = nn.BCEWithLogitsLoss()
    
    # Dataset and loader
    dataset = LoDoPaBDataset(TRAIN_OBS_DIR, TRAIN_GT_DIR)
    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # Training loop
    for epoch in range(EPOCHS):
        G.train()
        D.train()
        
        for i, (ldct, fdct) in enumerate(train_loader):
            ldct, fdct = ldct.to(DEVICE), fdct.to(DEVICE)
            
            # ---------------------
            # Train Discriminator
            # ---------------------
            opt_D.zero_grad()
            
            # Real images
            real_output = D(fdct)
            real_loss = criterion_bce(real_output, torch.ones_like(real_output))
            
            # Fake images
            fake = G(ldct).detach()
            fake_output = D(fake)
            fake_loss = criterion_bce(fake_output, torch.zeros_like(fake_output))
            
            # Total discriminator loss
            loss_D = (real_loss + fake_loss) / 2
            loss_D.backward()
            opt_D.step()
            
            # -----------------
            # Train Generator
            # -----------------
            opt_G.zero_grad()
            
            # Generate fake images
            fake = G(ldct)
            output = D(fake)
            
            # Adversarial loss
            adv_loss = criterion_bce(output, torch.ones_like(output))
            
            # Reconstruction loss (only if dimensions match)
            if fake.shape == fdct.shape:
                recon_loss = criterion_mse(fake, fdct)
            else:
                # If dimensions don't match, crop the larger one
                min_h = min(fake.size(2), fdct.size(2))
                min_w = min(fake.size(3), fdct.size(3))
                recon_loss = criterion_mse(fake[:, :, :min_h, :min_w], 
                                         fdct[:, :, :min_h, :min_w])
            
            # Total generator loss
            loss_G = adv_loss + 100 * recon_loss  # Weighted reconstruction loss
            loss_G.backward()
            opt_G.step()
            
            # Print progress
            if i % 10 == 0:
                print(f"Epoch {epoch+1}/{EPOCHS}, Batch {i}: G_loss={loss_G.item():.4f}, D_loss={loss_D.item():.4f}")
        
        # Save sample images
        with torch.no_grad():
            G.eval()
            ldct, fdct = next(iter(train_loader))
            ldct, fdct = ldct.to(DEVICE), fdct.to(DEVICE)
            fake = G(ldct)
            
            # Save images
            save_image(fake, f"{OUTPUT_DIR}/fake_epoch_{epoch+1}.png", normalize=True)
            save_image(fdct, f"{OUTPUT_DIR}/real_epoch_{epoch+1}.png", normalize=True)
            save_image(ldct, f"{OUTPUT_DIR}/input_epoch_{epoch+1}.png", normalize=True)
            
            # Save models
            torch.save(G.state_dict(), f"{CHECKPOINT_DIR}/generator_epoch_{epoch+1}.pth")
            torch.save(D.state_dict(), f"{CHECKPOINT_DIR}/discriminator_epoch_{epoch+1}.pth")

train()

Using device: mps
Epoch 1/50, Batch 0: G_loss=46.9108, D_loss=0.6792
Epoch 1/50, Batch 10: G_loss=8.0267, D_loss=0.6791
Epoch 1/50, Batch 20: G_loss=12.2159, D_loss=0.6119
Epoch 1/50, Batch 30: G_loss=8.5423, D_loss=0.6062
Epoch 1/50, Batch 40: G_loss=6.1380, D_loss=0.5454
Epoch 1/50, Batch 50: G_loss=2.4985, D_loss=0.6127
Epoch 1/50, Batch 60: G_loss=7.5722, D_loss=0.4982
Epoch 1/50, Batch 70: G_loss=4.2854, D_loss=0.6416
Epoch 1/50, Batch 80: G_loss=7.9827, D_loss=0.4703
Epoch 1/50, Batch 90: G_loss=8.3526, D_loss=0.4822
Epoch 1/50, Batch 100: G_loss=2.6826, D_loss=0.6530
Epoch 1/50, Batch 110: G_loss=6.6521, D_loss=0.4049
Epoch 1/50, Batch 120: G_loss=6.0910, D_loss=0.3936
Epoch 1/50, Batch 130: G_loss=7.5345, D_loss=0.3615
Epoch 1/50, Batch 140: G_loss=6.9781, D_loss=0.4677
Epoch 1/50, Batch 150: G_loss=6.3835, D_loss=0.4530
Epoch 1/50, Batch 160: G_loss=5.4160, D_loss=0.3558
Epoch 1/50, Batch 170: G_loss=6.8543, D_loss=0.4109
Epoch 1/50, Batch 180: G_loss=4.2157, D_loss=0.4288
Epo